# DBSCAN Experiments (Thesis-aligned) - Enhanced Version

Notebook ini fokus pada alur: 1) impor dan konfigurasi, 2) **duplicate analysis & normalization**, 3) k-distance plot untuk menentukan eps (dari unique samples jika perlu), 4) eksperimen parameter (eps & min_samples) menggunakan silhouette (sample), 5) fit final DBSCAN dan simpan model, 6) evaluasi cluster.

## 🆕 Enhanced Features:

1. **Cosine Distance Support** - Normalize embeddings untuk similarity yang lebih baik (especially untuk BERT embeddings)
2. **Smart Duplicate Handling** - Deteksi otomatis duplikat, compute eps dari unique samples, tapi fit DBSCAN pada FULL dataset (frequency preserved)
3. **Realistic Parameters** - min_samples 5-30 (bukan 3-12), KNN_NEIGHBORS = 20 (bukan 4)
4. **Better eps Range** - 75th-99th percentile (avoid excessive noise)
5. **Comprehensive Analysis** - Duplicate stats, k-distance dari unique samples, meaningful eps estimation

## 🎯 Problem Solved:

**Before:** k-distance = 0 for 90% of data (duplicates) → eps estimation fails  
**After:** k-distance computed from unique samples → meaningful eps → DBSCAN on full data (duplicates preserved for frequency info)

**Key Insight:** Duplicates = frequency information! Don't remove them, just compute eps smartly.


In [ ]:
# Part 1 — Imports & configuration
import numpy as np
from pathlib import Path
import joblib
import matplotlib.pyplot as plt
from sklearn.cluster import DBSCAN
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.preprocessing import normalize
import pandas as pd
import gc
from tqdm import tqdm
import time

# ============================================================================
# CONFIGURATION - Specify your embedding file(s) directly
# ============================================================================

# Option 1: Single file (BGL or Thunderbird)
INPUT_FILES = [
    Path("/media/bioinfo04/Expansion/2427051003_dataset_vector/after_preprocessed_bgl_embeddings.npy"),
    # Path("/media/bioinfo04/Expansion/2427051003_dataset_vector/after_preprocessed_thunderbird_embeddings.npy"),
]

# Option 2: Multiple files (Combined dataset)
# INPUT_FILES = [
#     Path("/media/bioinfo04/Expansion/2427051003_dataset_vector/after_preprocessed_bgl_embeddings.npy"),
#     Path("/media/bioinfo04/Expansion/2427051003_dataset_vector/after_preprocessed_thunderbird_embeddings.npy"),
# ]

# Option 3: PCA variants (smaller, faster - RECOMMENDED for DBSCAN!)
# INPUT_FILES = [
#     Path("/media/bioinfo04/Expansion/2427051003_dataset_vector_pca256/after_preprocessed_bgl_pca256_embeddings.npy"),
# ]

# Option 4: PCA128 for ultra-large datasets
# INPUT_FILES = [
#     Path("/media/bioinfo04/Expansion/2427051003_dataset_vector_pca128/after_preprocessed_thunderbird_pca128_embeddings.npy"),
# ]

RANDOM_STATE = 42
# ⚠️ MEMORY-OPTIMIZED: Reduced for 32GB RAM systems
SAMPLE_FOR_METRICS = 15000   # Reduced from 50k - for grid search (RAM intensive!)
SAMPLE_FOR_KDIST = 150000    # Reduced from 500k - for k-distance estimation
KNN_NEIGHBORS = 20           # for k-distance plot (k = min_samples, realistic for log embeddings)

# NEW: Duplicate & Distance handling
USE_COSINE_DISTANCE = True  # Normalize embeddings for cosine-like distance (recommended for embeddings)
HANDLE_DUPLICATES = True    # Compute k-distance from unique samples if many duplicates
DUPLICATE_THRESHOLD = 0.5   # If >50% duplicates, use unique samples for k-distance

print("📁 Input files configured:")
for f in INPUT_FILES:
    if f.exists():
        size_gb = f.stat().st_size / (1024**3)
        print(f"  ✓ {f.name} ({size_gb:.2f} GB)")
        if size_gb > 10:
            print(f"  ⚠️ WARNING: Large file ({size_gb:.1f}GB) detected!")
            print(f"     For 32GB RAM systems, consider using PCA variants:")
            print(f"     - PCA256: ~3.5GB (4× smaller, ~same quality)")
            print(f"     - PCA128: ~1.8GB (8× smaller, slight quality loss)")
    else:
        print(f"  ❌ NOT FOUND: {f}")

print(f"\n⚙️ Configuration:")
print(f"  SAMPLE_FOR_METRICS: {SAMPLE_FOR_METRICS:,} (grid search sample)")
print(f"  SAMPLE_FOR_KDIST: {SAMPLE_FOR_KDIST:,} (k-distance sample)")
print(f"  KNN_NEIGHBORS (min_samples): {KNN_NEIGHBORS}")
print(f"  Cosine distance: {'Enabled' if USE_COSINE_DISTANCE else 'Disabled'}")
print(f"  Duplicate handling: {'Enabled' if HANDLE_DUPLICATES else 'Disabled'}")
print(f"\n💡 Memory tips:")
print(f"  - If Part 3 crashes → reduce SAMPLE_FOR_METRICS to 10000")
print(f"  - For >10GB files → strongly recommend PCA256 or PCA128 variants")


## Test: File Detection & Size Analysis

Quick diagnostic untuk verify files dan estimate runtime.


In [ ]:
# Part 2 — Smart file loading with duplicate analysis and k-distance plot to estimate eps

def detect_embedding_dim(file_path: Path) -> int:
    """
    Auto-detect embedding dimension from filename pattern
    - *pca256* → 256 dims
    - *pca128* → 128 dims
    - default → 768 dims
    """
    filename = file_path.name.lower()
    if 'pca256' in filename:
        return 256
    elif 'pca128' in filename:
        return 128
    else:
        return 768

def infer_num_rows(path: Path, embedding_dim: int = None) -> int:
    """
    Infer number of rows for RAW memmap files
    Auto-detects dimension from filename if not provided
    """
    if embedding_dim is None:
        embedding_dim = detect_embedding_dim(path)
    size = path.stat().st_size
    return size // (embedding_dim * np.dtype(np.float32).itemsize)

def load_single_file_smart(file_path: Path, embedding_dim: int = None):
    """
    Smart loader: auto-detect .npy vs RAW memmap
    Returns (array, is_memmap, num_rows)
    
    Auto-detects dimension from filename if not provided:
    - *pca256* → 256 dims
    - *pca128* → 128 dims  
    - default → 768 dims
    """
    # Auto-detect dimension if not provided
    if embedding_dim is None:
        embedding_dim = detect_embedding_dim(file_path)
        print(f"   🔍 Auto-detected dimension: {embedding_dim} from filename")
    
    try:
        arr = np.load(file_path, mmap_mode='r')
        detected_dim = arr.shape[1]
        if detected_dim != embedding_dim:
            print(f"   ⚠️ Dimension mismatch! Expected {embedding_dim}, got {detected_dim} from .npy header")
            embedding_dim = detected_dim
        return arr, True, arr.shape[0]
    except Exception:
        num_rows = infer_num_rows(file_path, embedding_dim)
        arr = np.memmap(
            file_path, 
            dtype=np.float32, 
            mode='r', 
            shape=(num_rows, embedding_dim)
        )
        print(f"   ⚠️ Loaded as RAW memmap: {num_rows:,} rows × {embedding_dim} dims")
        return arr, True, num_rows

def load_embeddings_from_files(files, force_copy=False):
    """Load embeddings from list of file paths with auto-dimension detection"""
    if len(files) == 0:
        raise FileNotFoundError('No embedding files provided')
    
    for f in files:
        if not f.exists():
            raise FileNotFoundError(f'File not found: {f}')
    
    # Auto-detect dimension from first file
    first_arr, _, _ = load_single_file_smart(files[0])
    embedding_dim = first_arr.shape[1]
    print(f"Embedding dimension: {embedding_dim}")
    
    if len(files) == 1:
        print(f"Loading single file: {files[0].name}")
        return first_arr
    
    print(f"Loading {len(files)} files...")
    total_samples = 0
    file_info = []
    for f in files:
        arr, is_mmap, n_rows = load_single_file_smart(f)  # Auto-detect dimension
        file_info.append((f, arr, n_rows))
        total_samples += n_rows
        print(f"  - {f.name}: {n_rows:,} rows")
    
    total_size_gb = (total_samples * embedding_dim * 4) / (1024**3)
    print(f"\nTotal samples: {total_samples:,} ({total_size_gb:.2f} GB)")
    
    if total_size_gb < 100:
        print("Strategy: Memory-mapped stacking")
        arrays = [arr for _, arr, _ in file_info]
        return np.vstack(arrays)
    else:
        raise MemoryError(
            f"Dataset too large ({total_size_gb:.1f}GB). "
            "For DBSCAN, use single file or PCA variants (smaller size)"
        )

# Load embeddings
print("Loading embeddings...")
emb = load_embeddings_from_files(INPUT_FILES)
print(f'Loaded embeddings shape: {emb.shape}')

# NEW: Normalize for cosine distance
if USE_COSINE_DISTANCE:
    print('\n🔄 Normalizing embeddings for cosine-like distance...')
    emb = normalize(emb, norm='l2')
    print(f'✓ Embeddings normalized (using Euclidean on normalized = cosine distance)')

# NEW: Duplicate analysis
print('\n🔍 Analyzing duplicates...')
n_total = emb.shape[0]
if n_total > SAMPLE_FOR_KDIST:
    print(f'   Using sample of {SAMPLE_FOR_KDIST:,} for duplicate check')
    rng = np.random.RandomState(RANDOM_STATE)
    dup_check_idx = rng.choice(n_total, min(SAMPLE_FOR_KDIST, n_total), replace=False)
    emb_dup_check = emb[dup_check_idx]
else:
    emb_dup_check = emb
    dup_check_idx = None

unique_embeddings = np.unique(emb_dup_check, axis=0)
n_unique = len(unique_embeddings)
n_sample = len(emb_dup_check)
duplicate_ratio = 1 - (n_unique / n_sample)

print(f'\n📊 Duplicate Analysis (sample):')
print(f'   Total samples checked: {n_sample:,}')
print(f'   Unique samples: {n_unique:,}')
print(f'   Duplicate ratio: {duplicate_ratio:.2%}')

# Decide whether to use unique samples for k-distance
use_unique_for_kdist = HANDLE_DUPLICATES and duplicate_ratio > DUPLICATE_THRESHOLD

if use_unique_for_kdist:
    print(f'\n⚠️ High duplicate ratio detected ({duplicate_ratio:.1%} > {DUPLICATE_THRESHOLD:.0%})')
    print(f'   → Computing k-distance from UNIQUE samples only')
    print(f'   → eps will be meaningful (not 0)')
    print(f'   → Final DBSCAN will run on FULL dataset (with duplicates)')
    
    # Get unique samples from full dataset (or large sample)
    if n_total > SAMPLE_FOR_KDIST * 2:
        print(f'   Sampling {SAMPLE_FOR_KDIST:,} from full dataset first...')
        sample_idx = rng.choice(n_total, SAMPLE_FOR_KDIST, replace=False)
        emb_sample = emb[sample_idx]
        unique_for_kdist = np.unique(emb_sample, axis=0)
        del emb_sample, sample_idx
    else:
        print(f'   Extracting unique from full dataset...')
        unique_for_kdist = np.unique(emb, axis=0)
    
    print(f'   Unique samples for k-distance: {len(unique_for_kdist):,}')
    emb_for_kdist = unique_for_kdist
    del unique_embeddings
else:
    print(f'\n✅ Low duplicate ratio ({duplicate_ratio:.1%})')
    print(f'   → Using standard sampling for k-distance')
    # For very large datasets, sample for k-distance plot
    if n_total > SAMPLE_FOR_KDIST:
        print(f'   Using sample of {SAMPLE_FOR_KDIST:,} for k-distance plot')
        rng = np.random.RandomState(RANDOM_STATE)
        kdist_idx = rng.choice(n_total, SAMPLE_FOR_KDIST, replace=False)
        emb_for_kdist = emb[kdist_idx]
        del kdist_idx
    else:
        emb_for_kdist = emb
    del unique_embeddings

# Cleanup duplicate check variables
if dup_check_idx is not None:
    del emb_dup_check, dup_check_idx
gc.collect()

# Compute nearest-neighbors distances (k-distance)
print(f'\n🔄 Computing k-distance (k={KNN_NEIGHBORS})...')
print(f'   Sample size: {len(emb_for_kdist):,}')
nn = NearestNeighbors(n_neighbors=KNN_NEIGHBORS, n_jobs=-1, metric='euclidean')
nn.fit(emb_for_kdist)
distances, _ = nn.kneighbors(emb_for_kdist)
# distances[:, -1] is the distance to k-th neighbor
k_dist = np.sort(distances[:, -1])

# Cleanup nn object
del nn, distances
gc.collect()

# Filter out zeros if present
k_dist_nonzero = k_dist[k_dist > 0]
if len(k_dist_nonzero) < len(k_dist):
    pct_zero = (1 - len(k_dist_nonzero) / len(k_dist)) * 100
    print(f'   ⚠️ {pct_zero:.1f}% of k-distances are 0 (duplicates in unique sample)')
    print(f'   → Using non-zero distances only for statistics')
    k_dist_for_stats = k_dist_nonzero
else:
    k_dist_for_stats = k_dist

# Plot k-distance curve
plt.figure(figsize=(10,4))
plt.plot(k_dist)
plt.xlabel('Points sorted by k-distance')
plt.ylabel(f'k-distance (k={KNN_NEIGHBORS})')
title = 'k-distance plot — look for elbow to choose eps'
if use_unique_for_kdist:
    title += ' (computed from UNIQUE samples)'
plt.title(title)

if len(k_dist_for_stats) > 0:
    plt.axhline(y=np.percentile(k_dist_for_stats, 85), color='purple', linestyle=':', alpha=0.4, label='85th percentile')
    plt.axhline(y=np.percentile(k_dist_for_stats, 90), color='r', linestyle='--', alpha=0.5, label='90th percentile')
    plt.axhline(y=np.percentile(k_dist_for_stats, 95), color='orange', linestyle='--', alpha=0.5, label='95th percentile')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'\n📊 k-distance statistics:')
if len(k_dist_for_stats) > 0:
    # Display requested percentiles: 50, 60, 70, 80, 85, 90, 95
    percentiles = [50, 60, 70, 80, 85, 90, 95]
    for p in percentiles:
        val = np.percentile(k_dist_for_stats, p)
        print(f'   {p:2d}th percentile: {val:.4f}')
else:
    print('   ⚠️ All k-distances are 0!')
    print('   → Try: increase SAMPLE_FOR_KDIST, decrease KNN_NEIGHBORS, or enable normalization')

# Store k_dist_for_stats for next cell
k_dist_clean = k_dist_for_stats

# Cleanup k-distance variables
del k_dist, k_dist_nonzero
if 'emb_for_kdist' in locals() and emb_for_kdist is not emb:
    del emb_for_kdist
gc.collect()

print(f'\n✅ Part 2 complete. Memory cleaned up.')


In [ ]:
# Part 3 — Parameter search over eps and min_samples (uses a sample for silhouette)
# WARNING: DBSCAN can be slow for large datasets; we sample for metric computation
# ⚠️ MEMORY OPTIMIZED: Reduced grid + aggressive cleanup for 32GB RAM systems

print('Preparing sample for parameter search...')
n = emb.shape[0]
if n > SAMPLE_FOR_METRICS:
    rng = np.random.RandomState(RANDOM_STATE)
    sample_idx = rng.choice(n, SAMPLE_FOR_METRICS, replace=False)
    emb_sample = emb[sample_idx].copy()  # .copy() to avoid memmap issues
    print(f'Using sample: {SAMPLE_FOR_METRICS:,} / {n:,} samples')
    # Free memory
    del sample_idx
    gc.collect()
else:
    emb_sample = emb
    print(f'Using full dataset: {n:,} samples')

# Define parameter grid based on k-distance statistics
if len(k_dist_clean) > 0:
    eps_min = np.percentile(k_dist_clean, 75)  # Start higher to avoid too many noise points
    eps_max = np.percentile(k_dist_clean, 99)  # Extended to 99th percentile
    
    # If range is too small, expand it
    if eps_max - eps_min < 0.01:
        eps_min = np.percentile(k_dist_clean, 50)
        eps_max = np.percentile(k_dist_clean, 99)
    
    # ⚠️ REDUCED: 6 values instead of 8 to save memory
    eps_values = np.linspace(eps_min, eps_max, 6)
else:
    print('⚠️ No valid k-distances, using default eps range')
    eps_values = np.linspace(0.1, 2.0, 6)

# ⚠️ REDUCED: 3 values instead of 5 to save memory (18 combinations vs 40)
min_samples_values = [10, 20, 30]

print(f'\nParameter grid (MEMORY OPTIMIZED):')
print(f'  eps: {len(eps_values)} values from {eps_values.min():.4f} to {eps_values.max():.4f}')
print(f'  min_samples: {min_samples_values}')
print(f'  Total combinations: {len(eps_values) * len(min_samples_values)}')
print(f'  💡 For finer tuning later, increase grid around best config')

results = []

total_combinations = len(eps_values) * len(min_samples_values)
print(f'\n🔄 Testing {total_combinations} parameter combinations...')

pbar = tqdm(total=total_combinations, desc='DBSCAN grid search', unit='config')
for eps_idx, eps in enumerate(eps_values, 1):
    for ms in min_samples_values:
        start_time = time.time()
        
        # n_jobs=-1 to use all CPU cores
        dbs = DBSCAN(eps=float(eps), min_samples=int(ms), n_jobs=-1, metric='euclidean')
        labels = dbs.fit_predict(emb_sample)
        
        # Compute number of clusters (exclude noise label -1)
        unique_labels = set(labels) - {-1}
        n_clusters = len(unique_labels)
        n_noise = np.sum(labels == -1)
        noise_pct = (n_noise / len(labels)) * 100
        
        sil = -1
        if n_clusters > 1:
            try:
                # n_jobs=-1 to maximize CPU usage
                sil = silhouette_score(emb_sample, labels, n_jobs=-1)
            except Exception:
                sil = -1
        
        results.append({
            'eps': float(eps),
            'min_samples': int(ms),
            'n_clusters': n_clusters,
            'noise_pct': float(noise_pct),
            'silhouette': float(sil)
        })
        
        elapsed = time.time() - start_time
        pbar.update(1)
        tqdm.write(f'  eps={eps:.4g}, min_samples={ms:2d} → clusters={n_clusters:2d}, noise={noise_pct:5.1f}%, sil={sil:6.4f} ({elapsed:.1f}s)')
        
        # ⚠️ MEMORY CLEANUP: Free memory after each iteration
        del dbs, labels, unique_labels
        gc.collect()

pbar.close()

# Free sample memory before showing results
print('\n🧹 Cleaning up memory...')
del emb_sample
gc.collect()

# Show results sorted by silhouette
df_res = pd.DataFrame(results)
print('\n' + '='*70)
print('TOP 10 CONFIGURATIONS (by silhouette score)')
print('='*70)
print(df_res.sort_values('silhouette', ascending=False).head(10).to_string(index=False))

# Also show configurations with reasonable noise levels
print('\n' + '='*70)
print('CONFIGURATIONS WITH LOW NOISE (<30%)')
print('='*70)
df_low_noise = df_res[df_res['noise_pct'] < 30].sort_values('silhouette', ascending=False)
if len(df_low_noise) > 0:
    print(df_low_noise.head(10).to_string(index=False))
else:
    print('⚠️ No configurations with <30% noise found')
    print('   Consider adjusting eps range or min_samples values')


In [ ]:
# Part 4 — Fit final DBSCAN on full data and save model+labels

# Set chosen parameters based on Part 3 results
# Option 1: Best silhouette score
best_config = df_res.sort_values('silhouette', ascending=False).iloc[0]

# Option 2: Best with low noise (uncomment if preferred)
# df_low_noise = df_res[df_res['noise_pct'] < 30].sort_values('silhouette', ascending=False)
# if len(df_low_noise) > 0:
#     best_config = df_low_noise.iloc[0]
# else:
#     print('⚠️ No low-noise config, using best silhouette')
#     best_config = df_res.sort_values('silhouette', ascending=False).iloc[0]

CHOSEN_EPS = float(best_config['eps'])
CHOSEN_MIN_SAMPLES = int(best_config['min_samples'])

print('='*70)
print('FINAL DBSCAN CONFIGURATION')
print('='*70)
print(f'eps:         {CHOSEN_EPS:.6f}')
print(f'min_samples: {CHOSEN_MIN_SAMPLES}')
print(f'metric:      euclidean' + (' (on normalized data = cosine)' if USE_COSINE_DISTANCE else ''))
print(f'Expected clusters: {int(best_config["n_clusters"])}')
print(f'Expected noise:    {best_config["noise_pct"]:.1f}%')
print(f'Expected silhouette: {best_config["silhouette"]:.4f}')
print('='*70)

# ⚠️ Memory cleanup before final fit
print('\n🧹 Cleaning memory before final DBSCAN...')
if 'df_res' in locals():
    del df_res
if 'results' in locals():
    del results
if 'df_low_noise' in locals():
    del df_low_noise
gc.collect()

print(f'\n🔄 Fitting DBSCAN on FULL dataset ({emb.shape[0]:,} samples)...')
print('   ℹ️ Using FULL data (including duplicates)')
print('   ℹ️ Duplicates will automatically cluster together (distance = 0)')
print('⏳ This may take a while... Monitor CPU usage in Task Manager/htop')
start_time = time.time()
# n_jobs=-1 to use all CPU cores
model = DBSCAN(eps=CHOSEN_EPS, min_samples=CHOSEN_MIN_SAMPLES, n_jobs=-1, metric='euclidean')
labels_full = model.fit_predict(emb)
elapsed = time.time() - start_time
print(f'✓ DBSCAN completed in {elapsed/60:.1f} minutes')

# Save labels and model
out_model = Path('dbscan_model.pkl')
joblib.dump(model, out_model)
np.save('dbscan_labels.npy', labels_full)
np.save('dbscan_config.npy', np.array([CHOSEN_EPS, CHOSEN_MIN_SAMPLES, 1 if USE_COSINE_DISTANCE else 0]))

print(f'\n✓ Saved: {out_model}')
print(f'✓ Saved: dbscan_labels.npy ({len(labels_full):,} labels)')
print(f'✓ Saved: dbscan_config.npy (eps, min_samples, use_cosine)')

# Cleanup model object (keep only labels)
del model
gc.collect()


In [ ]:
# Part 5 — Evaluation and cluster analysis

from collections import Counter
counts = Counter(labels_full)

print('='*70)
print('CLUSTER ANALYSIS')
print('='*70)
print('\nLabel counts (including noise -1):')
for lbl, cnt in sorted(counts.items()):
    pct = (cnt / len(labels_full)) * 100
    cluster_type = 'NOISE' if lbl == -1 else f'Cluster {lbl}'
    print(f' - {cluster_type:12s}: {cnt:8,} samples ({pct:5.2f}%)')

# Separate noise and clusters
n_noise = counts.get(-1, 0)
n_clusters = len(set(labels_full) - {-1})
n_clustered = len(labels_full) - n_noise

print(f'\nSummary:')
print(f'  Total samples:    {len(labels_full):,}')
print(f'  Clusters found:   {n_clusters}')
print(f'  Clustered points: {n_clustered:,} ({(n_clustered/len(labels_full)*100):.1f}%)')
print(f'  Noise points:     {n_noise:,} ({(n_noise/len(labels_full)*100):.1f}%)')

# Compute metrics on sample or full dataset
n = emb.shape[0]
if n > SAMPLE_FOR_METRICS:
    print(f'\n📊 Computing metrics on sample ({SAMPLE_FOR_METRICS:,} samples)...')
    rng = np.random.RandomState(RANDOM_STATE)
    idx = rng.choice(n, SAMPLE_FOR_METRICS, replace=False)
    lbls_s = labels_full[idx]
    emb_s = emb[idx]
else:
    print(f'\n📊 Computing metrics on full dataset...')
    emb_s = emb
    lbls_s = labels_full

if len(set(lbls_s) - {-1}) > 1:
    try:
        sil = silhouette_score(emb_s, lbls_s)
        print(f'  Silhouette Score: {sil:.4f}')
    except Exception as e:
        print(f'  Silhouette Score: N/A ({e})')
    
    try:
        dbi = davies_bouldin_score(emb_s, lbls_s)
        print(f'  Davies-Bouldin Index: {dbi:.4f} (lower is better)')
    except Exception as e:
        print(f'  Davies-Bouldin Index: N/A ({e})')
else:
    print('⚠️ Not enough clusters (excluding noise) to compute metrics')

# Visualize cluster size distribution (excluding noise)
cluster_labels = [lbl for lbl in labels_full if lbl != -1]
if len(cluster_labels) > 0:
    cluster_counts = Counter(cluster_labels)
    plt.figure(figsize=(12, 4))
    
    plt.subplot(1, 2, 1)
    plt.bar(cluster_counts.keys(), cluster_counts.values())
    plt.xlabel('Cluster ID')
    plt.ylabel('Number of samples')
    plt.title('Cluster Size Distribution')
    plt.grid(True, alpha=0.3)
    
    plt.subplot(1, 2, 2)
    sizes = list(cluster_counts.values())
    plt.hist(sizes, bins=min(20, len(sizes)), edgecolor='black')
    plt.xlabel('Cluster size')
    plt.ylabel('Frequency')
    plt.title('Cluster Size Histogram')
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print(f'\nCluster size statistics:')
    print(f'  Min:    {min(sizes):,}')
    print(f'  Max:    {max(sizes):,}')
    print(f'  Mean:   {np.mean(sizes):,.1f}')
    print(f'  Median: {np.median(sizes):,.1f}')
